# Análise exploratória — caso Aroeira (Encontro 1)

Preparo pessoal antes do Encontro 1. Testa, com dado, as três justificativas do pedido da diretoria (C0-05) e levanta achados adicionais. Leitura narrativa dos resultados em `RESUMO_DESAFIO_1.md`.

**Não é** o Mapa Técnico do Problema nem a Hipótese Técnica — esses instrumentos são produzidos em squad, em sala, com proveniência citada célula a célula (`conjunto.campo`). Isto aqui é munição para chegar com hipóteses melhores que "o pedido como está".

**Como rodar**: se este notebook estiver na pasta `Projeto/Encontro 1/` junto da pasta `Dados/`, a célula abaixo lê os CSVs direto do disco. Se estiver no Google Colab, suba os 4 arquivos com `from google.colab import files; files.upload()` antes de rodar (mesmo procedimento do KIT-048) e ajuste `PASTA_DADOS` para `"."`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.width', 160)

PASTA_DADOS = Path('Dados')  # ajuste para Path('.') no Colab, apos o upload

d1 = pd.read_csv(PASTA_DADOS / 'd1_frota.csv', parse_dates=['data_inicio_garantia'])
d2 = pd.read_csv(PASTA_DADOS / 'd2_telemetria.csv', parse_dates=['data'])
d3 = pd.read_csv(PASTA_DADOS / 'd3_ordens.csv', parse_dates=['data_abertura'])
d4 = pd.read_csv(PASTA_DADOS / 'd4_garantia.csv', parse_dates=['data'])

for nome, df in [('d1_frota', d1), ('d2_telemetria', d2), ('d3_ordens', d3), ('d4_garantia', d4)]:
    print(f'{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas')

## Visão geral da frota e das ordens

Antes de testar hipóteses, vale conferir a composição básica: quantos equipamentos, de que modelo, com que tipo de contrato, e como as ordens se dividem por tipo.

In [ ]:
print('Modelos:')
print(d1['modelo'].value_counts())
print()
print('Tipo de contrato:')
print(d1['tipo_contrato'].value_counts())
print()
print('Com telemetria:', d1['tem_telemetria'].value_counts().to_dict())
print()
print('Tipo de ordem (D3):')
print(d3['tipo_ordem'].value_counts())

## Validação de integridade dos dados

Confere as regras de consistência descritas na camada 0 (C0-07 a C0-10) antes de confiar em qualquer análise — ex.: equipamento sem telemetria não deveria aparecer em D2, pedido de garantia só deveria existir sobre ordem corretiva.

In [ ]:
avisos = []

divergente = d1[d1['ano_instalacao'] != d1['data_inicio_garantia'].dt.year]
if len(divergente):
    avisos.append(f'{len(divergente)} equipamento(s) com ano_instalacao != ano de data_inicio_garantia')

sem_telemetria = set(d1.loc[d1['tem_telemetria'] == 0, 'id_equipamento'])
aparecem_em_d2 = set(d2['id_equipamento']) & sem_telemetria
if aparecem_em_d2:
    avisos.append(f'{len(aparecem_em_d2)} equipamento(s) sem telemetria aparecem em D2')

campos_medicao = ['horas_operacao', 'ciclo_trabalho_medio', 'pressao_descarga_media',
                   'temp_descarga_media', 'temp_descarga_maxima', 'contagem_partidas']
incompletas = d2[d2['transmissao_incompleta'] == 1]
if incompletas[campos_medicao].notna().any(axis=1).sum():
    avisos.append('ha linha(s) com transmissao_incompleta=1 mas com alguma medicao preenchida')

ordens_corretivas_ids = set(d3.loc[d3['tipo_ordem'] == 'corretiva', 'id_ordem'])
fora_de_corretiva = d4[~d4['id_ordem'].isin(ordens_corretivas_ids)]
if len(fora_de_corretiva):
    avisos.append(f'{len(fora_de_corretiva)} pedido(s) de garantia sobre ordem que nao e corretiva')

print('nenhum desvio encontrado.' if not avisos else '\n'.join(avisos))

## Hipótese 1 — "o tempo médio de atendimento subiu no painel de serviço"

Olhar duração média por atendimento corretivo **e** volume de ordens corretivas, mês a mês. Se a duração individual ficou quase estável mas o volume cresceu muito, o sintoma "tempo subindo" pode ser efeito de fila/capacidade, não de atendimento mais lento.

In [ ]:
corr = d3[d3['tipo_ordem'] == 'corretiva'].copy()
corr['mes'] = corr['data_abertura'].dt.to_period('M')

tendencia = corr.groupby('mes').agg(duracao_media_horas=('duracao_horas', 'mean'), n_ordens=('id_ordem', 'count'))
print(tendencia)

x = np.arange(len(tendencia))
inclinacao = np.polyfit(x, tendencia['duracao_media_horas'].values, 1)[0]
crescimento_volume = tendencia['n_ordens'].iloc[-1] / tendencia['n_ordens'].iloc[0]
print(f'\ninclinacao da duracao media: {inclinacao:.4f} h/mes')
print(f'crescimento do volume (ultimo mes / primeiro mes): {crescimento_volume:.1f}x')

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(tendencia.index.astype(str), tendencia['duracao_media_horas'], color='tab:blue', marker='o')
ax1.set_ylabel('duracao media (h)', color='tab:blue')
ax1.set_xlabel('mes')
ax1.tick_params(axis='x', rotation=45)

ax2 = ax1.twinx()
ax2.bar(tendencia.index.astype(str), tendencia['n_ordens'], alpha=0.25, color='tab:orange')
ax2.set_ylabel('numero de ordens corretivas', color='tab:orange')

plt.title('Duracao media x volume de corretivas, por mes')
fig.tight_layout()
plt.show()

**Leitura candidata**: a duração por atendimento sobe pouco (~8% no ano); o volume de corretivas triplica. O quadro de técnicos (próxima seção) não cresce no período — gargalo de capacidade é candidato mais forte que "atendimento individual mais lento". D3 não tem um campo de tempo de espera/fila — essa é uma lacuna de dado a registrar, não uma suposição a fechar aqui.

## Hipótese 2 — "entraram técnicos sem experiência na série CP"

Sem um campo de data de contratação, a melhor proxy disponível é a data da primeira ordem de cada técnico. Testar se técnicos que aparecem depois têm desempenho pior (duração maior, mais causa não identificada).

In [ ]:
tecnicos = corr.groupby('id_tecnico').agg(
    primeira_ordem=('data_abertura', 'min'),
    n_ordens=('id_ordem', 'count'),
    duracao_media=('duracao_horas', 'mean'),
    prop_causa_nao_identificada=('causa_registrada', lambda s: (s == 'causa_nao_identificada').mean()),
).sort_values('primeira_ordem')
tecnicos['rank_entrada'] = tecnicos['primeira_ordem'].rank()
print(tecnicos)
print()
print('correlacao rank_entrada x duracao_media:', round(tecnicos[['rank_entrada','duracao_media']].corr().iloc[0,1], 3))

**Leitura candidata**: os 14 técnicos aparecem todos entre 01/set e 02/out/2025 — não há um grupo claramente "mais novo" entrando depois no período coberto pelos dados. A correlação entre ordem de entrada e duração média é ≈0. A afirmação da diretoria pode ser verdadeira, mas **não é testável com os dados disponíveis** — é uma lacuna a declarar no Mapa (dimensão "dados e qualidade"), não um fato a aceitar ou negar sem evidência.

## Hipótese 3 — "reincidência de chamados no mesmo equipamento"

Para cada equipamento, olhar pares consecutivos de ordem corretiva: quanto tempo passou entre uma e outra, e se repetem a mesma causa. Reincidência "quente" (mesma causa em até 30 dias) sugere reparo que não eliminou a causa raiz.

In [ ]:
corr_ordenado = corr.sort_values(['id_equipamento', 'data_abertura'])

linhas = []
for equipamento, grupo in corr_ordenado.groupby('id_equipamento'):
    grupo = grupo.reset_index(drop=True)
    for i in range(1, len(grupo)):
        delta_dias = (grupo.loc[i, 'data_abertura'] - grupo.loc[i - 1, 'data_abertura']).days
        linhas.append({
            'id_equipamento': equipamento,
            'causa': grupo.loc[i, 'causa_registrada'],
            'delta_dias': delta_dias,
            'mesma_causa': grupo.loc[i, 'causa_registrada'] == grupo.loc[i - 1, 'causa_registrada'],
        })

pares = pd.DataFrame(linhas)
pares['reincidencia_quente'] = pares['mesma_causa'] & (pares['delta_dias'] <= 30)

print(f"pares consecutivos no mesmo equipamento: {len(pares)}")
print(f"mesma causa que o atendimento anterior: {pares['mesma_causa'].sum()} ({pares['mesma_causa'].mean()*100:.1f}%)")
quente = pares[pares['reincidencia_quente']]
print(f"reincidencia 'quente' (mesma causa em ate 30 dias): {len(quente)} ({len(quente)/len(pares)*100:.1f}%)")
print()
print(quente['causa'].value_counts())

In [ ]:
quente['causa'].value_counts().plot(kind='barh', figsize=(7,4), color='tab:red')
plt.title("Reincidencia 'quente' (mesma causa em ate 30 dias) por causa")
plt.xlabel('numero de pares')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Leitura candidata**: reincidência confirmada e concentrada em 3 causas (`falha_eletrica`, `vazamento_oleo`, `contaminacao_particulado`) — as três causas mais frequentes do dataset inteiro. Aponta para eficácia do reparo nessas causas específicas, não para falta de informação do técnico no momento do atendimento.

## Achado adicional — a hipótese ambiental mais óbvia não se sustenta

Seria razoável esperar que `contaminacao_particulado` fosse mais comum em plantas de `classe_ambiente = particulado_alto`. Checar antes de assumir.

In [ ]:
corr_ambiente = corr.merge(d1[['id_equipamento', 'classe_ambiente']], on='id_equipamento', how='left')
tabela_ambiente = pd.crosstab(corr_ambiente['classe_ambiente'], corr_ambiente['causa_registrada'], normalize='index')
print(tabela_ambiente['contaminacao_particulado'].round(3))

**Leitura**: contra-intuitivo — `particulado_alto` (22,8%) tem proporção **menor** de `contaminacao_particulado` que `particulado_baixo` (30,5%). Vale desconfiar de explicações "óbvias" sem checar o dado — é exatamente o tipo de achado que vale citar com proveniência (`D1.classe_ambiente` x `D3.causa_registrada`) no Mapa.

## Achado adicional — telemetria como sinal de alerta antes da corretiva

Para equipamentos com telemetria, comparar a temperatura máxima de descarga nos 7 dias antes do atendimento contra uma linha de base (30–60 dias antes). Um delta positivo e consistente por causa é candidato a sinal de manutenção preditiva.

In [ ]:
equipamentos_com_telemetria = set(d1.loc[d1['tem_telemetria'] == 1, 'id_equipamento'])
corr_telemetria = corr[corr['id_equipamento'].isin(equipamentos_com_telemetria)]

resultados = []
for _, ordem in corr_telemetria.iterrows():
    equipamento, abertura = ordem['id_equipamento'], ordem['data_abertura']
    recente = d2[(d2['id_equipamento'] == equipamento) & (d2['data'] < abertura) &
                 (d2['data'] >= abertura - pd.Timedelta(days=7))]
    base = d2[(d2['id_equipamento'] == equipamento) &
              (d2['data'] < abertura - pd.Timedelta(days=30)) &
              (d2['data'] >= abertura - pd.Timedelta(days=60))]
    if len(recente) >= 3 and len(base) >= 3:
        resultados.append({
            'causa': ordem['causa_registrada'],
            'delta_temp_max': recente['temp_descarga_maxima'].mean() - base['temp_descarga_maxima'].mean(),
        })

alerta = pd.DataFrame(resultados).groupby('causa').agg(
    delta_temp_max_medio=('delta_temp_max', 'mean'),
    n_amostras=('delta_temp_max', 'count'),
).round(2).sort_values('delta_temp_max_medio', ascending=False)
print(alerta)

In [ ]:
alerta['delta_temp_max_medio'].plot(kind='barh', figsize=(7,4), color='tab:green')
plt.title('Delta de temperatura maxima (7 dias antes vs. linha de base) por causa')
plt.xlabel('delta de temperatura (graus C)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Leitura**: `falha_valvula_admissao` e `falha_vedacao` mostram alta de temperatura antes do atendimento — mas com poucas amostras (n pequeno, ver contagem acima), tratar como pista. As 3 causas mais frequentes e mais reincidentes (`falha_eletrica`, `vazamento_oleo`, `contaminacao_particulado`) têm amostra grande e **não** mostram esse aviso prévio — são falhas mais súbitas, telemetria de temperatura não ajuda a prevê-las. Manutenção preditiva via telemetria é, no máximo, uma solução **parcial**.

## Achado adicional — panorama de garantia

Fora do pedido original, mas um ponto de atrito que pode reaparecer na Matriz de Riscos/Validação mais adiante.

In [ ]:
print('distribuicao das decisoes:')
print(d4['decisao_cobertura'].value_counts(normalize=True).round(3))
print()
print('motivo por decisao:')
print(pd.crosstab(d4['decisao_cobertura'], d4['motivo_decisao']))
print()
print('taxa de contestacao do cliente, por decisao:')
print(d4.groupby('decisao_cobertura')['contestado_pelo_cliente'].mean().round(3))

**Leitura**: 34% dos pedidos são negados, majoritariamente por "uso fora de especificação"; a taxa de contestação do cliente é 5x maior numa decisão negativa (20,6%) do que numa total (4%) — um ponto de atrito real, ainda que fora do escopo direto do pedido da diretoria.

## Síntese — reframe para levar ao squad

O pedido original mira **acesso à informação durante o atendimento** (assistente conversacional). Os dados sugerem leituras alternativas mais prováveis para cada sintoma citado:

- *tempo subindo* → possível **gargalo de capacidade/fila** (volume 3,2x, equipe estável), não falta de informação;
- *técnico novo sem experiência* → **não testável** com os dados atuais, é lacuna de dado, não fato;
- *reincidência* → **real, mas concentrada em 3 causas**, sugerindo problema de eficácia do reparo (ou de diagnóstico) nessas causas, não de acesso a manual.

Candidatas de IA a levar para o Mapa Técnico do Problema (sem escolher nenhuma — regra do Encontro 1):
1. Manutenção preditiva seletiva via telemetria (causas com sinal de alerta: válvula/vedação).
2. Apoio ao diagnóstico de causa raiz nas 3 causas reincidentes.
3. Dimensionamento/alocação de capacidade de atendimento.
4. Módulo de apoio informacional ao técnico (o pedido original) — agora como uma candidata entre outras.

Ver `RESUMO_DESAFIO_1.md` para a versão narrativa completa, e `analise/` para o código reutilizável fora deste notebook.